In [ ]:
# ============================================================
# Imports & globale Konfiguration
# Imports & global configuration
# ============================================================

#TODO: Muss auf ECG Datensatz angepasst werden!


import numpy as np
import torch

from utils.models.anoformer import (
    RANDOM_SEED,
    DEVICE,
    TranADDataset,
    TranADMainModel,
    set_seed,
)
from utils.explainability.attention import TranADExplainability

set_seed(RANDOM_SEED)
print(f"Active device: {DEVICE}")

In [ ]:
# ============================================================
# Demo: Synthetische Zeitreihe mit Anomalien
# Vollständiger Durchlauf / Complete end-to-end run
# Schritt / Step: Daten → DataLoader → Modell → Training →
#                 Erkennung → Evaluation → Explainability
# ============================================================

print("  TranAD Demo – Synthetic Time Series")


# ── 1. Synthetische Zeitreihe erzeugen / Create synthetic time series ──
RNG         = np.random.default_rng(RANDOM_SEED)  # Reproduzierbarer Zufallsgenerator / Reproducible RNG
TotalTime   = 3000                                 # 3000 Zeitpunkte / 3000 timesteps
NumFeatures = 5                                    # 5 parallele Kanäle / 5 parallel channels
TimeAxis    = np.linspace(0, 10 * np.pi, TotalTime)  # Gleichmäßige Zeitachse 0..10π / Uniform time axis

# Fünf verschiedene periodische Signale + Gauss-Rauschen / Five periodic signals + Gaussian noise
RawData = np.column_stack([
    np.sin(TimeAxis)                       + RNG.normal(0, 0.05, TotalTime),   # F0: einfacher Sinus / simple sine
    np.cos(2.3 * TimeAxis)                 + RNG.normal(0, 0.07, TotalTime),   # F1: Kosinus höherer Frequenz / higher-freq cosine
    0.6 * np.sin(3.7 * TimeAxis + 1.0)    + RNG.normal(0, 0.06, TotalTime),   # F2: Oberton / overtone
    0.4 * np.cos(0.5 * TimeAxis)           + RNG.normal(0, 0.04, TotalTime),   # F3: langsame Drift / slow drift
    np.sin(TimeAxis) * np.cos(TimeAxis)    + RNG.normal(0, 0.05, TotalTime),   # F4: Mischung beider / mix of both
]).astype(np.float32)                              # float32 für PyTorch / float32 for PyTorch

TrueLabels = np.zeros(TotalTime, dtype=np.int64)   # Alle Punkte zunächst normal / All points initially normal

# Sechs Anomalie-Segmente / Six anomaly segments
AnomalySegments = [
    (200,  220),   # Kurzer Spike am Anfang / Short spike at start
    (650,  680),   # Drift-Anomalie / Drift anomaly
    (1050, 1070),  # Mittlere Anomalie / Medium anomaly
    (1450, 1490),  # Breites Anomalie-Fenster / Wide anomaly window
    (1900, 1915),  # Schmaler Spike / Narrow spike
    (2650, 2690),  # Abschließende Anomalie / Final anomaly
]

for (AnomalyStart, AnomalyEnd) in AnomalySegments:
    Magnitude = RNG.uniform(2.5, 4.5, (AnomalyEnd - AnomalyStart, NumFeatures))
    RawData[AnomalyStart:AnomalyEnd, :] += Magnitude    # Großen Versatz injizieren / Inject large offset
    TrueLabels[AnomalyStart:AnomalyEnd]  = 1            # Segment als anomal markieren / Mark as anomalous

print(f"Total timesteps  : {TotalTime}")
print(f"Anomalous points : {TrueLabels.sum()}")
print(f"Number of features: {NumFeatures}")
print(f"Anomaly segments : {len(AnomalySegments)}")

# ── 2. Train/Test-Split ──────────────────────────────────────────────
SplitIndex   = int(TotalTime * 0.75)          # 75% Training, 25% Test
TrainingData = RawData[:SplitIndex]            # Erste 2250 Punkte / First 2250 points
TestData     = RawData[SplitIndex:]            # Letzte 750 Punkte / Last 750 points
TestLabels   = TrueLabels[SplitIndex:]         # Labels nur für Testbereich / Labels only for test range
print(f"\nTraining length: {len(TrainingData)} |  Test length: {len(TestData)}")

# ── 3. TranADDataLoader erstellen / Create TranADDataLoader ─────────
WindowSize  = 64  # Jedes Fenster hat 64 Zeitschritte / Each window has 64 timesteps
DataManager = TranADDataLoader(
    TrainingData       = TrainingData,
    TestData           = TestData,
    WindowSize         = WindowSize,
    TrainingStride     = 4,          # Überlappende Fenster beim Training / Overlapping training windows
    TestStride         = None,       # Standard: kein Überlapp (= WindowSize) / Default: no overlap
    NormalizationType  = "zscore",   # Z-Score-Normalisierung / Z-score normalization
    BatchSize          = 32,
    TestLabels         = TestLabels, # Labels dem Test-Dataset mitgeben / Pass labels to test dataset
)

# ── 4. TranADMainModel erstellen / Create TranADMainModel ───────────
TranAD = TranADMainModel(
    NumFeatures         = NumFeatures,   # Muss mit Daten übereinstimmen / Must match data
    ModelDimension      = 32,            # Kleine Dimension für schnellen Demo-Lauf / Small dim for fast demo
    NumHeads            = 2,             # Zwei Attention-Köpfe / Two attention heads
    NumEncoderLayers    = 2,             # Flacher Encoder / Shallow encoder
    NumDecoderLayers    = 2,             # Flache Decoder / Shallow decoders
    ForwardDimension    = 64,            # d_ff = 2 × d_model / d_ff = 2 × d_model
    DropoutRate         = 0.1,           # 10% Dropout / 10% dropout
    LearningRate        = 5e-4,          # Leicht erhöhte LR für schnellere Konvergenz / Slightly higher LR
    FocusGrowthRate     = 1.3,           # φ wächst 30% pro Epoche / φ grows 30% per epoch
    InitialFocusScore   = 1.0,           # φ startet bei 1 / φ starts at 1
    MaxFocusScore       = 50.0,          # φ maximal 50 / φ maximum 50
    AnomalyPercentile   = 95.0,          # 95. Perzentil als Anomalie-Schwelle / 95th percentile as threshold
)

# ── 5. Training ──────────────────────────────────────────────────────
print("\n── Training starts ─────────────────────────────")
TranAD.TrainModel(
    TrainingLoader = DataManager.TrainingLoader,
    NumEpochs      = 30,    # 30 Epochen für Demo / 30 epochs for demo
    PrintInterval  = 5,     # Fortschrittsausgabe alle 5 Epochen / Progress every 5 epochs
)

# ── 6. Anomalie-Erkennung auf Testdaten / Anomaly detection on test data ─
print("\n── Anomaly Detection ──────────────────────────")
AnomalyScores, BinaryPredictions = TranAD.DetectAnomalies(DataManager.TestLoader)
print(f"Detected anomalous windows: {BinaryPredictions.sum()}")

# ── 7. Evaluation mit Ground-Truth / Evaluation with ground-truth ────
print("\n── Evaluation ──────────────────────────────────────────────────────")
# Window-Labels ableiten: Fenster gilt als anomal wenn ≥1 Punkt anomal
WindowLabels = np.array([
    TestLabels[i * WindowSize:(i + 1) * WindowSize].max()  # Max = 1 wenn irgendein Punkt anomal / max=1 if any anomalous
    for i in range(len(DataManager.TestDataset))
])
Metrics = TranAD.Evaluate(DataManager.TestLoader, WindowLabels)

# ── 8. Explainability / Explainability ────────────────────────────────
print("\n── Explainability ──────────────────────────────────────────────────")
Explainer = TranADExplainability(TranAD)   # Explainer mit Modellreferenz / Explainer with model reference

# 8a. Trainingsverlauf: Loss und φ / Training history: loss and φ
Explainer.ShowTrainingHistory()

# 8b. Erstes Testfenster holen / Get first test window
FirstWindow, _ = DataManager.TestDataset[0]               # (T, F)
FirstWindow    = FirstWindow.unsqueeze(0).to(TranAD.Device)  # → (1, T, F)

# 8c. Phase-1 vs. Phase-2 Vergleich für Feature 0 / Phase-1 vs. Phase-2 comparison
Explainer.ShowPhaseComparison(
    FirstWindow, FeatureIndex=0, DatasetForInverseNorm=DataManager.TestDataset
)

# 8d. Rekonstruktionsfehler-Visualisierung / Reconstruction error visualization
with torch.no_grad():
    _, Rec2 = TranAD.Network(FirstWindow)               # Phase-2-Rekonstruktion / Phase-2 reconstruction
OrigNP = DataManager.TestDataset.InverseNormalization(FirstWindow[0].cpu().numpy())  # (T, F) Originalskala
RecNP  = DataManager.TestDataset.InverseNormalization(Rec2[0].cpu().numpy())          # (T, F)

Explainer.ShowReconstructionError(
    OriginalData      = OrigNP,
    ReconstructedData = RecNP,
    AnomalyScores     = AnomalyScores,
    TrueLabels        = WindowLabels,
    Threshold         = TranAD.AnomalyThreshold,
)

# 8e. Attention-Heatmap für Encoder Layer 0 / Attention heatmap for encoder layer 0
Explainer.ShowAttentionHeatmap(FirstWindow, LayerIndex=0, HeadIndex=0, Component="Encoder")

# 8f. Feature-Wichtigkeit mit Saliency / Feature importance with saliency
Explainer.ShowFeatureImportance(
    FirstWindow,
    FeatureNames=["Sinus", "Kosinus", "Oberton", "Drift", "Kreuzprodukt"],
)

# 8g. Focus-Score-Effekt / Focus score effect
Explainer.ShowFocusScoreEffect(FirstWindow, FocusScoreList=[0.5, 1.0, 2.0, 5.0, 10.0, 30.0])

print("\n TranAD demo complete.")